# Real receipts: evaluate before training

The baseline for the real receipts from Kaggle ([dhiaznaidi/receiptdatasetssd300v2](https://www.kaggle.com/datasets/dhiaznaidi/receiptdatasetssd300v2) by default), before any training on them. The notebook downloads the dataset with your Kaggle key, turns it into suites with `real_receipts.py`, and scores the models below on `dev-receipts`, the 15% of the receipts no training will see. The gold is audited along the way. `real_receipts_sft.ipynb` trains on `train-receipts` after this, and its result is measured against these rows.

The rows, each `<name>-dev-receipts`:

| run | model | served by | why |
|---|---|---|---|
| `qwen3.5-0.8b-grpo-r2-q8_0` | the shipped file, [kurexi/cost-0.8b-q8](https://huggingface.co/kurexi/cost-0.8b-q8) | llama-server, as in `baseline_gguf_t4.ipynb` | **the baseline**: what the app does with real receipts today |
| `qwen3.5-0.8b-grpo-r2-q4_k_m` | [kurexi/cost-0.8b-q4_k_m](https://huggingface.co/kurexi/cost-0.8b-q4_k_m) | llama-server | the smaller file beside it |
| `qwen3.5-0.8b-fp16` | `Qwen/Qwen3.5-0.8B`, untuned | vLLM, fp16 | the floor: what distillation started from |
| `qwen3.8-27b-fp8-v5` | the teacher, `Qwen/Qwen3.8-27B-FP8` | vLLM | the ceiling, and the gold audit: a case the teacher gets wrong is often a wrong label |

**The shipped file is checked first.** Its SHA-256 must match the one recorded when its `dev` row was scored. With `CHECK_DEV`, it is also run on `dev` on this GPU and compared with the committed row, so a different server or GPU can't pass for a change in the model.

**Then the gold is audited.** `audit.py` counts, field by field, what each model gets wrong, and shows the cases the teacher misses beside every model's answer. When every model misses the same field, the gold is the first suspect. If the gold is wrong, fix the settings or the converter and build new suites before training on them. A baseline on a wrong gold says nothing.

**Every run is kept on Drive** and resumes after a disconnect. Rerunning the notebook skips the rows already finished. Add a model to `MODELS` (a GGUF file, or a Hugging Face folder for vLLM) to score it on the same cases later, for example a round of `real_receipts_sft.ipynb` once it's quantized.

**The GPU.** An **A100** runs every row, because the 27B teacher needs more than 40 GB. On an L4 or a T4 the teacher's row is skipped and the 0.8B rows still run.

**Your Kaggle key.** Put it in Colab's secrets (the key icon on the left) as `KAGGLE_USERNAME` and `KAGGLE_KEY`, or as `KAGGLE_API_TOKEN` for a new-style `KGAT_…` token, and allow this notebook to read them. With no secret set, the cell asks for the key in a hidden prompt. Either way it is passed only to the `kaggle` download command.

The receipts and the suites stay on Drive and out of git, because the dataset's licence decides what may be published. The zip at the end holds the runs' `meta.json`, `results.jsonl` and `grade.json`, which quote the receipts' merchants in their drafts, so check the licence before committing them.

In [ ]:
!nvidia-smi

In [ ]:
import os
import subprocess
from google.colab import userdata

def sh(args, cwd=None, env=None):
    process = subprocess.Popen(args, cwd=cwd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                               bufsize=1)
    for line in process.stdout:
        print(line, end="", flush=True)
    if process.wait():
        raise subprocess.CalledProcessError(process.returncode, args[:3])


def secret(name):
    try:
        return userdata.get(name) or ""
    except Exception:
        return ""


REPO_URL = "https://github.com/kurexi/cost"
BRANCH = "claude/modest-brahmagupta-hpbdfb"
RECORDS = "eval-records"
AGENT = ""
CLONE = "/content/repo"
EVAL = os.path.join(CLONE, AGENT, "eval")
RECORDS_DIR = f"{CLONE}/{RECORDS}" if RECORDS else EVAL
token = secret("GITHUB_TOKEN")
url = REPO_URL.replace("https://", f"https://{token}@", 1) if token else REPO_URL
os.chdir("/content")
sh(["rm", "-rf", CLONE])
sh(["git", "clone", "--depth", "1", "--branch", BRANCH, url, CLONE])
os.environ["SPENDING_EVAL_RECORDS"] = RECORDS_DIR
os.makedirs(f"{RECORDS_DIR}/runs", exist_ok=True)

## Settings

**The dataset** (the same in `real_receipts_eval.ipynb` and `real_receipts_sft.ipynb`; the suites are built once, and a notebook whose settings differ from the suites on Drive stops). `DATASET` is the Kaggle dataset. Set `CURRENCY` for receipts that print no currency of their own: the converter recognises `RM`, `€`, `£`, `US$`, `S$`, `Rp` and other markers, and uses `CURRENCY` only where it finds none. A bare `$` names no country (Malaysian receipts print it for ringgit), so it is read as `CURRENCY` too. Categories come from keyword rules on the shop's name and the receipt's words, in English, French and Malay, and every receipt of one shop gets the same category. A merchant is right under the label's legal name, a trading name its receipts print (`TRADING_NAMES` in `real_receipts.py`, "Bar Wang Rice" for Unihakka International), the name without a branch in brackets, or the spelling of the same shop on another receipt. A credit note is a refund, so its gold is not spending. `MONTH_FIRST` reads `03/04/2019` as March 4th instead of April 3rd. `LANGUAGE` picks the category titles (`auto`: French if most receipts read French, otherwise English). With `OCR = "auto"`, tesseract runs only when the dataset gives no text; set `"always"` or `"never"` to force it. `OCR_LANGUAGES` lists tesseract's languages, for example `eng+fra`.

**The models.** `MODELS` maps a run name to how it is served:
- `gguf`: a Hugging Face repo and file, or a path to a `.gguf` file on Drive, served by llama-server. `sha256` checks the file.
- `vllm`: a Hugging Face id or a folder, served by vLLM with `dtype`. `teacher: True` marks a model that needs an A100.

`CHECK_DEV` names the GGUF row that is also run on `dev` and compared with the committed `dev` run of that name. `AUDIT` is the run the audit takes its cases from: the teacher if it ran, otherwise the shipped model.

In [ ]:
DATASET = "dhiaznaidi/receiptdatasetssd300v2"
CURRENCY = "MYR"
MONTH_FIRST = False
LANGUAGE = "auto"
OCR = "auto"
OCR_LANGUAGES = "eng+fra"
SEED = 51
EVAL_SHARE = 0.15
TRAIN_COPIES = 2
TRAIN_SUITE = "train-receipts"
EVAL_SUITE = "dev-receipts"
TEACHER_NAME = "qwen3.8-27b-fp8-v5"

MODELS = {
    "qwen3.5-0.8b-grpo-r2-q8_0": {"gguf": ("kurexi/cost-0.8b-q8", "qwen3.5-0.8b-grpo-r2-q8_0.gguf"),
                                  "sha256": "88355e5ee1d18d3aa2590cdb63d6d29488a702541f12bc2ace4b420a826cf4c5"},
    "qwen3.5-0.8b-grpo-r2-q4_k_m": {"gguf": ("kurexi/cost-0.8b-q4_k_m", "qwen3.5-0.8b-grpo-r2-q4_k_m.gguf"),
                                    "sha256": "ed3dbc0474927a68e5bde9adece4b15c925c5081134d370bd9f4933a0428be26"},
    "qwen3.5-0.8b-fp16": {"vllm": "Qwen/Qwen3.5-0.8B", "dtype": "float16"},
    TEACHER_NAME: {"vllm": "Qwen/Qwen3.8-27B-FP8", "teacher": True},
}
SHIPPED = "qwen3.5-0.8b-grpo-r2-q8_0"
CHECK_DEV = SHIPPED
AUDIT = TEACHER_NAME

## Drive

In [ ]:
import json
import shutil
from google.colab import drive

drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/spending-agent"
DATA_ZIP = f"{DRIVE}/data/{DATASET.split('/')[-1]}.zip"
RAW = f"/content/data/{DATASET.split('/')[-1]}"
SUITES_KEPT = f"{DRIVE}/suites"
for folder in (os.path.dirname(DATA_ZIP), SUITES_KEPT, f"{DRIVE}/runs"):
    os.makedirs(folder, exist_ok=True)


def keep_on_drive(run):
    kept, local = f"{DRIVE}/runs/{run}", f"{RECORDS_DIR}/runs/{run}"
    os.makedirs(kept, exist_ok=True)
    if os.path.islink(local):
        return
    if os.path.isdir(local) and not os.listdir(kept):
        shutil.copytree(local, kept, dirs_exist_ok=True)
    sh(["rm", "-rf", local])
    os.symlink(kept, local)


def finished(folder, cases):
    results = f"{folder}/results.jsonl"
    return os.path.exists(results) and sum(1 for _ in open(results)) >= cases

## The receipts from Kaggle

The zip is downloaded once into `MyDrive/spending-agent/data/` and unpacked to local disk on every run. Unpacking thousands of small files onto Drive would be slow.

The key comes from Colab's secrets (`KAGGLE_USERNAME` and `KAGGLE_KEY`, or `KAGGLE_API_TOKEN`). Without them, the cell asks for it in a hidden prompt. It is handed only to the `kaggle` command's environment, and is never written to disk or kept on Drive.

In [ ]:
import getpass

if not os.path.exists(DATA_ZIP):
    username, key, api_token = secret("KAGGLE_USERNAME"), secret("KAGGLE_KEY"), secret("KAGGLE_API_TOKEN")
    if not api_token and not (username and key):
        print("No Kaggle secret in this notebook's Colab secrets: type it here instead.")
        typed = getpass.getpass("Kaggle API key, or a KGAT_ API token: ").strip()
        if typed.startswith("KGAT_"):
            api_token = typed
        else:
            username, key = input("Kaggle username: ").strip(), typed
    kaggle_env = {k: v for k, v in os.environ.items() if not k.startswith("KAGGLE_")}
    if api_token:
        kaggle_env["KAGGLE_API_TOKEN"] = api_token
    if username and key:
        kaggle_env.update(KAGGLE_USERNAME=username, KAGGLE_KEY=key)
    sh(["pip", "install", "-q", "-U", "kaggle"])
    download = "/content/kaggle-download"
    sh(["rm", "-rf", download])
    sh(["kaggle", "datasets", "download", DATASET, "-p", download], env=kaggle_env)
    del kaggle_env, username, key, api_token
    zips = [name for name in os.listdir(download) if name.endswith(".zip")]
    if len(zips) != 1:
        raise SystemExit(f"expected one zip from Kaggle, found {os.listdir(download)}")
    shutil.copy(f"{download}/{zips[0]}", DATA_ZIP)
    sh(["rm", "-rf", download])
if not os.path.isdir(RAW):
    os.makedirs(RAW)
    sh(["unzip", "-q", "-o", DATA_ZIP, "-d", RAW])
print(DATA_ZIP, f"{os.path.getsize(DATA_ZIP) / 1e6:.1f} MB")

## What the dataset holds

File types, the largest folders, and the start of one file of each annotation type. If the *Convert* cell keeps no receipts, this output says why. Paste it into a Claude Code session to have the converter taught the layout.

In [ ]:
from collections import Counter

extensions, folders, samples = Counter(), Counter(), {}
for folder, _, names in os.walk(RAW):
    for name in names:
        ext = os.path.splitext(name)[1].lower() or "(none)"
        extensions[ext] += 1
        folders[os.path.relpath(folder, RAW)] += 1
        if ext in (".txt", ".json", ".jsonl", ".csv", ".xml", ".tsv") and ext not in samples:
            samples[ext] = os.path.join(folder, name)
print("files by type:", dict(extensions.most_common()))
print("folders:")
for folder, count in folders.most_common(15):
    print(f"  {count:6d}  {folder}")
for ext, path in samples.items():
    print(f"\n== {os.path.relpath(path, RAW)}")
    with open(path, encoding="utf-8", errors="replace") as f:
        print(f.read(1200))

## Convert

The suites are built once and kept on Drive (`MyDrive/spending-agent/suites/`); a rerun copies them back, and stops if they were built with other settings than these. The counts say how many receipts were read, by which layout (`files:`), and why the others were dropped (`dropped:`). Three cases follow, with their gold records.

**Check them.** The labels come from the dataset, and the currency, the date order and the category come from the settings and from keyword rules. If a case is wrong (a currency that is never printed, say, or US-style dates), change the setting, delete `MyDrive/spending-agent/suites/train-receipts` and `dev-receipts`, and run this cell again. Changing them after runs have started is a new suite: `run.py` refuses to resume on a different fingerprint, and the earlier runs can't be compared with the new ones.

**Text read on a Mac.** Run `ocr_macos.py` on a Mac (macOS 13 or later) to read the photos with macOS's own OCR: `uv run --with pyobjc-framework-Vision python eval/ocr_macos.py --images <dataset>/fullDataset/images --out <folder>`. It writes one `.txt` per photo, one line per text observation in Vision's order, as the app's OCR hands text to the agent (`--layout lines` joins the observations printed side by side instead), and skips photos already read. Upload its folder of `.txt` files to `MyDrive/spending-agent/data/<dataset>-macos-ocr/` (for this dataset, `receiptdatasetssd300v2-macos-ocr`). Where the dataset gives no text, the converter takes it from there instead of running tesseract here.

In [ ]:
os.chdir(EVAL)
MACOS_TEXT = f"{DRIVE}/data/{DATASET.split('/')[-1]}-macos-ocr"
TEXT_DIR = MACOS_TEXT if os.path.isdir(MACOS_TEXT) and os.listdir(MACOS_TEXT) else None
print(f"text read on a Mac: {len(os.listdir(TEXT_DIR))} files in {TEXT_DIR}" if TEXT_DIR else
      f"no text read on a Mac ({MACOS_TEXT})")
if "--text-dir" not in subprocess.run(["python3", "real_receipts.py", "--help"], capture_output=True,
                                      text=True).stdout:
    raise SystemExit(f"the real_receipts.py cloned from {BRANCH} is older than this notebook: push "
                     "eval/real_receipts.py, eval/ocr_macos.py and spending_agent/money.py, then run from the top")


def convert(ocr):
    args = ["python3", "real_receipts.py", "--source", RAW, "--dataset", DATASET, "--train-name", TRAIN_SUITE,
            "--eval-name", EVAL_SUITE, "--seed", str(SEED), "--eval-share", str(EVAL_SHARE), "--currency", CURRENCY,
            "--language", LANGUAGE, "--train-copies", str(TRAIN_COPIES), *(["--month-first"] if MONTH_FIRST else []),
            *(["--text-dir", TEXT_DIR] if TEXT_DIR else []),
            *(["--ocr", "--ocr-languages", OCR_LANGUAGES] if ocr else [])]
    try:
        sh(args)
        return True
    except subprocess.CalledProcessError:
        return False


def install_tesseract():
    languages = [l for l in OCR_LANGUAGES.split("+") if l != "eng"]
    sh(["apt-get", "-qq", "install", "-y", "tesseract-ocr", *[f"tesseract-ocr-{l}" for l in languages]])
    sh(["pip", "install", "-q", "pytesseract", "pillow"])


if all(os.path.exists(f"{SUITES_KEPT}/{name}/meta.json") for name in (TRAIN_SUITE, EVAL_SUITE)):
    for name in (TRAIN_SUITE, EVAL_SUITE):
        shutil.copytree(f"{SUITES_KEPT}/{name}", f"suites/{name}", dirs_exist_ok=True)
    print("suites from Drive")
else:
    images = sum(n for ext, n in extensions.items() if ext in (".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"))
    ok = OCR != "always" and convert(False)
    if ok and OCR == "auto" and images:
        with open(f"suites/{TRAIN_SUITE}/meta.json") as f:
            ok = json.load(f)["receipts"] >= 0.3 * images * (1 - EVAL_SHARE)
        if not ok:
            print("few receipts came with text: reading the images with tesseract instead")
    if not ok and OCR != "never" and images and (OCR == "always" or not TEXT_DIR):
        install_tesseract()
        ok = convert(True)
    if not ok:
        raise SystemExit("no usable receipts: see the counts above and the output of the cell before")
    for name in (TRAIN_SUITE, EVAL_SUITE):
        shutil.copytree(f"suites/{name}", f"{SUITES_KEPT}/{name}", dirs_exist_ok=True)

SUITE_CASES = {}
expected = {"dataset": DATASET, "seed": SEED, "evalShare": EVAL_SHARE, "dayFirst": not MONTH_FIRST,
            "defaultCurrency": CURRENCY, "languageSetting": LANGUAGE, "trainCopies": TRAIN_COPIES}
for name in (TRAIN_SUITE, EVAL_SUITE):
    with open(f"suites/{name}/meta.json") as f:
        meta = json.load(f)
    differs = {k: (meta.get(k), v) for k, v in expected.items() if meta.get(k) != v}
    if (OCR == "never" and meta.get("ocr")) or (OCR == "always" and not meta.get("ocr")):
        differs["ocr"] = (meta.get("ocr"), OCR)
    if bool(meta.get("textDir")) != bool(TEXT_DIR):
        differs["textDir"] = (bool(meta.get("textDir")), bool(TEXT_DIR))
    if differs:
        raise SystemExit(f"the suites on Drive were built with other settings (on Drive, here): {differs}. "
                         f"Use their settings, or delete {SUITES_KEPT}/{TRAIN_SUITE} and {SUITES_KEPT}/{EVAL_SUITE} "
                         "to build new suites (a new suite: earlier runs on them can't be compared)")
    SUITE_CASES[name] = meta["cases"]
    print(name, {k: meta[k] for k in ("cases", "users", "receipts", "fingerprint", "currencies", "language", "ocr")})
print("conversion:", json.dumps(meta["conversion"], indent=1))
with open(f"suites/{EVAL_SUITE}/cases.jsonl", encoding="utf-8") as f:
    shown = [json.loads(line) for line in f][:3]
for case in shown:
    gold = case["gold"]
    print("\n" + "=" * 80 + f"\n{case['id']}  {case['traits']}\n" + case["content"][:1500])
    print("-> gold:", {k: gold[k] for k in ("merchant", "totalAmount", "currencyCode", "purchaseDate", "purchaseTime",
                                            "category")})

## What is still to run

Each row on `dev-receipts` resumes from Drive. The teacher is skipped where the GPU is too small for it.

In [ ]:
os.chdir(EVAL)
gpu_mib = int(subprocess.run(["nvidia-smi", "--query-gpu=memory.total", "--format=csv,noheader,nounits"],
                             capture_output=True, text=True).stdout.split()[0])
TEACHER_FITS = gpu_mib >= 39000
RUN_NAMES = {name: f"{name}-{EVAL_SUITE}" for name in MODELS}
TODO = {}
for name, spec in MODELS.items():
    keep_on_drive(RUN_NAMES[name])
    if finished(f"{RECORDS_DIR}/runs/{RUN_NAMES[name]}", SUITE_CASES[EVAL_SUITE]):
        print(f"{RUN_NAMES[name]}: done")
    elif spec.get("teacher") and not TEACHER_FITS:
        print(f"{RUN_NAMES[name]}: skipped, this GPU has {gpu_mib} MiB and the teacher needs an A100")
    else:
        TODO[name] = spec
        print(f"{RUN_NAMES[name]}: to run")
CHECK_RUN = f"{CHECK_DEV}-check-dev" if CHECK_DEV else None
if CHECK_RUN:
    keep_on_drive(CHECK_RUN)
for name, backend in (("check-oracle", "oracle"), ("check-silent", "silent")):
    graded = subprocess.run(["python3", "run.py", "--suite", EVAL_SUITE, "--name", name, "--backend", backend,
                             "--restart"], capture_output=True, text=True).stdout
    print(name, [line for line in graded.splitlines() if line.startswith("record_core")])

## The vLLM rows

The teacher and the untuned 0.8B, with the teacher's `dev` settings: thinking off, 16 cases in flight, the default limits. The vLLM install has its own virtual environment. The teacher's weights are about 30 GB, so its first start downloads for several minutes. Logs go to `/content/vllm.log`.

In [ ]:
VLLM_VENV = "/content/vllm-venv"
if any("vllm" in spec for spec in TODO.values()):
    if not os.path.exists(f"{VLLM_VENV}/bin/vllm"):
        sh(["pip", "install", "-q", "uv"])
        sh(["uv", "venv", "-q", "--python", "3.12", VLLM_VENV])
        sh(["uv", "pip", "install", "-q", "--python", f"{VLLM_VENV}/bin/python", "vllm"])
    print(subprocess.run([f"{VLLM_VENV}/bin/python", "-c", "import vllm; print(vllm.__version__)"],
                         capture_output=True, text=True).stdout)
else:
    print("no vLLM row to run")

In [ ]:
import time
import urllib.request


def stop_servers():
    subprocess.run("pkill -f '[v]llm serve'; pkill -f '[l]lama-server'", shell=True)
    for _ in range(60):
        used = subprocess.run(["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"],
                              capture_output=True, text=True).stdout.strip()
        if int(used.split()[0]) < 1000:
            return
        time.sleep(5)
    raise SystemExit(f"the GPU still holds {used} MiB")


def log_tail(path, characters=4000):
    with open(path, errors="replace") as f:
        return f.read()[-characters:]


def wait_for(process, log_path, name, limit_seconds=2400):
    started = reported = time.time()
    while True:
        if process.poll() is not None:
            raise SystemExit(f"{name} exited with code {process.returncode}:\n{log_tail(log_path)}")
        try:
            urllib.request.urlopen("http://127.0.0.1:8090/v1/models", timeout=5)
            print(f"{name} is up after {time.time() - started:.0f} s")
            return
        except Exception:
            pass
        if time.time() - started > limit_seconds:
            raise SystemExit(f"{name} isn't up after {limit_seconds} s:\n{log_tail(log_path)}")
        if time.time() - reported >= 30:
            lines = log_tail(log_path, 1000).strip().splitlines()
            print(f"{time.time() - started:.0f} s, still starting: {lines[-1] if lines else '(no log yet)'}", flush=True)
            reported = time.time()
        time.sleep(2)


def serve(model, extra=""):
    stop_servers()
    log_path = "/content/vllm.log"
    process = subprocess.Popen([f"{VLLM_VENV}/bin/vllm", "serve", model, "--served-model-name", "qwen",
                                "--max-model-len", "16384", "--max-num-seqs", "32", "--gpu-memory-utilization", "0.90",
                                "--enable-auto-tool-choice", "--tool-call-parser", "qwen3_coder",
                                "--reasoning-parser", "qwen3", "--port", "8090", *extra.split()],
                               stdout=open(log_path, "w"), stderr=subprocess.STDOUT, start_new_session=True)
    wait_for(process, log_path, "vLLM")


RUN_ARGS = ["--base-url", "http://127.0.0.1:8090/v1", "--model", "qwen", "--reasoning-effort", "none", "--workers", "16"]


def check_server():
    os.chdir(EVAL)
    probe = subprocess.run(["python3", "probe_formats.py", "--base-url", "http://127.0.0.1:8090/v1", "--model", "qwen"],
                           capture_output=True, text=True)
    print(probe.stdout, probe.stderr)
    verdicts = dict((line[:32].strip(), line[32:].strip()) for line in probe.stdout.splitlines() if line.strip())
    for required in ("tool round (tool_choice auto)", "full bounded schema"):
        if not verdicts.get(required, "").startswith("accepted"):
            raise SystemExit(f"the server refused the {required}: not a fair run, stop here")

In [ ]:
os.chdir(EVAL)
for name, spec in TODO.items():
    if "vllm" not in spec:
        continue
    print(f"== {RUN_NAMES[name]}", flush=True)
    serve(spec["vllm"], f"--dtype {spec['dtype']}" if spec.get("dtype") else "")
    check_server()
    sh(["python3", "run.py", "--suite", EVAL_SUITE, "--name", RUN_NAMES[name], *RUN_ARGS])
stop_servers()

## llama.cpp

Release `b12400`, the build the shipped file was scored with on `dev`: its prebuilt Linux CUDA 12.8 binaries, or a build from source if the driver can't run them or the server doesn't see the GPU. The server's flags are those of `baseline_gguf_t4.ipynb`: every layer on the GPU, 16 slots of 16,384 tokens, Jinja templates, and four context checkpoints a slot.

In [ ]:
LLAMA_TAG = "b12400"
LLAMA = "/content/llama.cpp"
ASSET = f"llama-{LLAMA_TAG}-bin-ubuntu-cuda-12.8-x64"
PREBUILT = f"/content/llama-{LLAMA_TAG}"
RELEASE = f"https://github.com/ggml-org/llama.cpp/releases/download/{LLAMA_TAG}"
os.environ["CUDA_CACHE_MAXSIZE"] = str(4 << 30)
NEEDS_LLAMA = any("gguf" in spec for spec in TODO.values()) or (
    CHECK_RUN and not finished(f"{RECORDS_DIR}/runs/{CHECK_RUN}", 280))


def sees_cuda(bin_dir):
    out = subprocess.run([f"{bin_dir}/llama-server", "--list-devices"], capture_output=True, text=True)
    return "CUDA0" in out.stdout + out.stderr


def prebuilt():
    driver = int(subprocess.run(["nvidia-smi", "--query-gpu=driver_version", "--format=csv,noheader"],
                                capture_output=True, text=True).stdout.strip().split(".")[0])
    if driver < 570:
        print(f"driver {driver} is older than CUDA 12.8 needs")
        return None
    try:
        if not os.path.exists(f"{PREBUILT}/llama-server"):
            for name in (ASSET, f"cudart-{ASSET}"):
                sh(["curl", "-fsSL", "-o", f"/content/{name}.tar.gz", f"{RELEASE}/{name}.tar.gz"])
                sh(["tar", "-xzf", f"/content/{name}.tar.gz", "-C", "/content"])
            subprocess.run(f"cp -a /content/cudart-{ASSET}/. {PREBUILT}/", shell=True, check=True)
    except subprocess.CalledProcessError as error:
        print(f"the release download failed: {error}")
        return None
    return PREBUILT if sees_cuda(PREBUILT) else None


def built():
    if not os.path.exists(f"{LLAMA}/build/bin/llama-server"):
        if not os.path.exists(f"{LLAMA}/CMakeLists.txt"):
            sh(["rm", "-rf", LLAMA])
            sh(["git", "clone", "-q", "--depth", "1", "--branch", LLAMA_TAG, "https://github.com/ggml-org/llama.cpp", LLAMA])
        sh(["cmake", "-B", "build", "-DGGML_CUDA=ON", "-DCMAKE_CUDA_ARCHITECTURES=native", "-DLLAMA_OPENSSL=OFF",
            "-DLLAMA_BUILD_TESTS=OFF", "-DLLAMA_BUILD_EXAMPLES=OFF"], cwd=LLAMA)
        sh(["cmake", "--build", "build", "--config", "Release", "-j", str(os.cpu_count()), "--target", "llama-server"],
           cwd=LLAMA)
    if not sees_cuda(f"{LLAMA}/build/bin"):
        raise SystemExit("the built server doesn't see the GPU either: check Runtime → Change runtime type")
    return f"{LLAMA}/build/bin"


if NEEDS_LLAMA:
    BIN = prebuilt()
    BINARIES = "release" if BIN else "built from source"
    BIN = BIN or built()
    print(f"llama.cpp {LLAMA_TAG}, {BINARIES}: {BIN}")
else:
    print("no GGUF row to run")

In [ ]:
import hashlib
from huggingface_hub import hf_hub_download


def gguf_file(spec):
    path = spec["gguf"] if isinstance(spec["gguf"], str) else hf_hub_download(*spec["gguf"], local_dir="/content/models")
    digest = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            digest.update(block)
    if spec.get("sha256") and digest.hexdigest() != spec["sha256"]:
        raise SystemExit(f"{path} has SHA-256 {digest.hexdigest()}, not {spec['sha256']}: not the file that was scored")
    return path, digest.hexdigest()


SERVER_FLAGS = ["-ngl", "all", "-np", "16", "-c", "262144", "--jinja", "-cram", "2048", "-ctxcp", "4"]


def serve_gguf(path):
    stop_servers()
    log_path = "/content/llama-server.log"
    process = subprocess.Popen([f"{BIN}/llama-server", "-m", path, "--alias", "qwen", *SERVER_FLAGS,
                                "--host", "127.0.0.1", "--port", "8090"],
                               stdout=open(log_path, "w"), stderr=subprocess.STDOUT, start_new_session=True)
    wait_for(process, log_path, "llama-server", 1200)


def record_server(run, path, digest):
    served = {"source": path, "llama_cpp_tag": LLAMA_TAG, "llama_cpp_binaries": BINARIES, "gguf_bytes": os.path.getsize(path),
              "gguf_sha256": digest, "server": "llama-server " + " ".join(SERVER_FLAGS),
              "gpu": subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], capture_output=True,
                                    text=True).stdout.strip()}
    with open(f"{RECORDS_DIR}/runs/{run}/server.json", "w") as f:
        json.dump(served, f, indent=1)


os.chdir(EVAL)
for name, spec in MODELS.items():
    if "gguf" not in spec or (name not in TODO and not (name == CHECK_DEV and CHECK_RUN)):
        continue
    path, digest = gguf_file(spec)
    print(f"== {name}: {path} ({os.path.getsize(path) / 1e9:.3f} GB, sha256 {digest[:16]}…)", flush=True)
    serve_gguf(path)
    check_server()
    if name == CHECK_DEV and not finished(f"{RECORDS_DIR}/runs/{CHECK_RUN}", 280):
        sh(["python3", "run.py", "--suite", "dev", "--name", CHECK_RUN, *RUN_ARGS])
        record_server(CHECK_RUN, path, digest)
    if name in TODO:
        sh(["python3", "run.py", "--suite", EVAL_SUITE, "--name", RUN_NAMES[name], *RUN_ARGS])
        record_server(RUN_NAMES[name], path, digest)
stop_servers()

## The shipped file on `dev`, here against where it was scored

The committed `dev` run of `CHECK_DEV` (a T4) beside this GPU's run of the same file, on the same 280 cases. The two should agree within the noise: a few cases flipped either way, `record_core` within about ±0.03. If they don't, the server or GPU changed what the model sees, and the `dev-receipts` rows must be read with that in mind.

In [ ]:
os.chdir(EVAL)
if CHECK_RUN and finished(f"{RECORDS_DIR}/runs/{CHECK_RUN}", 280):
    sh(["python3", "compare.py", CHECK_DEV, CHECK_RUN])

## The baseline on `dev-receipts`

Every finished row side by side, the shipped model first, with paired-bootstrap intervals and the score by trait:
- `known_merchant` and `new_merchant`: whether the user had filed the merchant before;
- `category_*`: what kind of shop it is;
- `ocr_from_image`: text read here by tesseract;
- `date_not_matched_in_text`: the labelled date wasn't found printed as a date.
- `total_not_in_text`: the labelled total isn't printed legibly, or at all, so it has to be worked out from the other lines.
- `credit_note`: a refund, which the agent should mark as not spending.

Then, for each row, the drafts the app would file without review and how many of them are right.

In [ ]:
os.chdir(EVAL)
order = [SHIPPED, *(name for name in MODELS if name != SHIPPED)]
present = [RUN_NAMES[name] for name in order if finished(f"{RECORDS_DIR}/runs/{RUN_NAMES[name]}", SUITE_CASES[EVAL_SUITE])]
if len(present) >= 2:
    sh(["python3", "compare.py", *present])
for run in present:
    with open(f"{RECORDS_DIR}/runs/{run}/grade.json") as f:
        graded = json.load(f)
    overall, per_case = graded["summary"]["overall"], graded["per_case"]
    approved = [c for c in per_case if c["would_approve"]]
    print(f"{run}: record_core {overall['record_core']:.3f}  record_all {overall['record_all']:.3f}  "
          f"false approvals {overall['false_approval']:.3f}  approval precision "
          f"{sum(c['record_all'] for c in approved) / max(1, len(approved)):.2f} over {len(approved)} approved")

## Audit the gold

The first table counts, field by field, the cases each run gets wrong; the `all wrong` row counts the cases every run misses on that field. Then the cases `AUDIT` gets wrong, with every run's answer beside the gold, most-missed first.

Read twenty or so before trusting the baseline. Typical faults, and what fixes them:
- **The currency is never printed**, and every model gives the same one: set `CURRENCY`.
- **The day and month are swapped**: set `MONTH_FIRST`.
- **The label is the legal name and the receipt prints a brand** (McDonald's for Gerbang Alaf Restaurants): add the brand to `TRADING_NAMES` in `real_receipts.py`. The merchant check already accepts any name inside the other, so a miss that isn't a brand is usually an OCR error.
- **The category is wrong for the shop**: the keyword rules in `real_receipts.py` (`CATEGORY_WORDS`) misread its name.
- **The total is the subtotal, or a tip is missing**: a label error in the dataset. Add the right value to `LABEL_FIXES` in `real_receipts.py`, keyed by the label's company, date and total.

`--field amount` (or `merchant`, `date`, ...) shows only that field's misses.

In [ ]:
os.chdir(EVAL)
audited = [RUN_NAMES[name] for name in (AUDIT, SHIPPED, *MODELS) if name in MODELS]
audited = list(dict.fromkeys(r for r in audited if finished(f"{RECORDS_DIR}/runs/{r}", SUITE_CASES[EVAL_SUITE])))
if audited:
    sh(["python3", "audit.py", *audited, "--limit", "25"])

## Take the results home

`board.py --write` adds the `dev-receipts` rows to `RESULTS.md`. The zip holds, for every run, `meta.json`, `results.jsonl`, `grade.json` and `server.json`, plus the suites' `meta.json` (the conversion counts and fingerprints), and `RESULTS.md`. Transcripts stay on Drive.

In [ ]:
os.chdir(EVAL)
sh(["python3", "board.py", "--write"])
OUT = "/content/out"
shutil.rmtree(OUT, ignore_errors=True)
for run in [*RUN_NAMES.values(), CHECK_RUN]:
    folder = f"{RECORDS_DIR}/runs/{run}" if run else None
    if folder and finished(folder, 1):
        os.makedirs(f"{OUT}/runs/{run}")
        for name in ("meta.json", "results.jsonl", "grade.json", "server.json"):
            if os.path.exists(f"{folder}/{name}"):
                shutil.copy(f"{folder}/{name}", f"{OUT}/runs/{run}/")
for name in (TRAIN_SUITE, EVAL_SUITE):
    os.makedirs(f"{OUT}/suites/{name}")
    shutil.copy(f"suites/{name}/meta.json", f"{OUT}/suites/{name}/")
shutil.copy(f"{RECORDS_DIR}/RESULTS.md", OUT)
subprocess.run(f"cd {OUT} && zip -qr /content/{EVAL_SUITE}-baseline.zip .", shell=True, check=True)
from google.colab import files
files.download(f"/content/{EVAL_SUITE}-baseline.zip")